# QniNotebook チュートリアル

このノートブックでは、QURI Partsで作った小規模回路について、Pythonコード、回路構造、各ステップまでの状態ベクトルを対応付けて確認します。

このデモでは、探索や振幅増幅につながる1回のGrover反復を題材にします。条件`111`を位相でマークし、拡散演算で位相差を測定確率へ変換してから測定します。実際のQURI Parts利用者がどの場面でこの確認方法を必要とするかは、今後の利用者との議論で検証します。

## このチュートリアルで確認すること
- QURI Partsで5量子ビットのGrover探索回路を作る
- `qni.show_circuit(circuit)` で回路構造を確認する
- `qni.show_circuit_and_state(circuit)` で回路と状態ベクトルを表示する
- ステップ境界を選び、操作に伴う状態変化を確認する
- 位相オラクル、拡散演算、測定の各境界を確認する

> 初期デモは1〜8量子ビットの読み取り専用確認に限定します。GUI編集や `commit()`、大規模回路は扱いません。

## 準備

Qniの表示機能と、QURI Partsの量子回路を読み込みます。

In [ ]:
from importlib import reload
from qni_jupyter import qni

# 同じカーネルに古いQniが残っていても、現在の実装を読み直す
qni.close()
qni = reload(qni)

from quri_parts.circuit import QuantumCircuit

## 1. まずは回路を作る

3量子ビットの候補 $q_0q_1q_2$ から`111`を探索します。$q_3$は条件フラグ、$q_4$は条件計算用の作業量子ビットです。位相オラクルの後に拡散演算を1回適用し、$q_0$〜$q_2$を同じ番号の古典ビットへ測定します。

In [ ]:
def build_grover_circuit():
    result = QuantumCircuit(5, cbit_count=5)

    # q0〜q2: 候補状態を一様な重ね合わせにする
    for qubit in range(3):
        result.add_H_gate(qubit)

    # q4へ q0 AND q1 を計算する
    result.add_TOFFOLI_gate(0, 1, 4)

    # q3へ q0 AND q1 AND q2 を計算する
    result.add_TOFFOLI_gate(4, 2, 3)

    # 条件111の振幅だけ位相を反転する
    result.add_Z_gate(3)

    # 補助量子ビットq3、q4を|0⟩へ戻す
    result.add_TOFFOLI_gate(4, 2, 3)
    result.add_TOFFOLI_gate(0, 1, 4)

    # 位相差を測定確率の差へ変換する
    for qubit in range(3):
        result.add_H_gate(qubit)
    for qubit in range(3):
        result.add_X_gate(qubit)
    result.add_H_gate(2)
    result.add_TOFFOLI_gate(0, 1, 2)
    result.add_H_gate(2)
    for qubit in range(3):
        result.add_X_gate(qubit)
    for qubit in range(3):
        result.add_H_gate(qubit)

    # 探索対象の3量子ビットを測定する
    result.measure([0, 1, 2], [0, 1, 2])
    return result


circuit = build_grover_circuit()

circuit

## 2. 回路構造を確認する

まず `qni.show_circuit()` で、状態準備、位相オラクル、拡散演算、測定の順序と制御線、およびセル実行時に得た測定結果を確認します。状態ベクトルパネルは表示しません。

In [ ]:
qni.show_circuit(circuit)

## 3. 回路と中間状態で確認する内容

`qni.show_circuit_and_state()` は、回路と状態ベクトルを並べて表示します。左の回路で選択したステップ境界と、右の状態は常に同じ計算時点を表します。表示する前に、各境界で期待する変化を確認します。

## 4. ステップごとの状態を確認する

この回路では、位相オラクルで付けた印が拡散演算によって確率差へ変わり、その後の測定へつながります。回路の各ステップ境界を選び、コード上の操作、回路図上の位置、その時点までの状態ベクトルを対応付けて確認します。

この確認方法が、挙動確認、技術説明、学習、問題調査のどの場面で役立つかは、デモ後の利用者評価で検証します。

次のセルを実行したら、回路の各ステップ境界を示す**縦棒**を左から順に選び、次の変化を確認します。

1. 最初のHステップで、候補 `000`〜`111`が一様な重ね合わせになる
2. 位相オラクル内のZで、条件`111`に対応する振幅だけ位相がπ変化する
3. オラクル末尾で、補助量子ビット$q_3$と$q_4$が $|0\rangle$へ戻る
4. 拡散演算途中のHでは、破壊的干渉により`000`、`001`、`010`の振幅が一時的に0になる
5. 拡散演算後、`111`の確率が78.125%、それ以外が各3.125%になる
6. 測定ゲートには、このセル実行に対応する1 shotの結果が表示される

状態ベクトルの基底ラベルは $q_4q_3q_2q_1q_0$ の順です。測定直前は$q_3=q_4=0$なので、`00111`が探索対象`111`に対応します。測定は確率的であり、1回の実行で必ず`111`になるわけではありません。同じセル出力内では測定結果が固定され、ステップ境界を選んでも変わりません。セルを再実行したときだけ変わる可能性があります。

In [ ]:
qni.show_circuit_and_state(circuit)

## 5. 研究ワークフローへのつながり

この例は、探索・振幅増幅で使われるオラクルや拡散演算を、より大きな回路へ組み込む前の確認へ展開できます。特に次の観点をNotebook上で確認できます。

- 意図した候補だけに位相変化が入っているか
- 条件計算に使った補助量子ビットが $|0\rangle$へ戻っているか
- 位相差が拡散演算によって意図した確率差へ変換されているか
- 測定が対象量子ビットと同じ番号の古典ビットへ対応しているか
- サブルーチンを前後の回路へ組み込む前に、局所的な挙動を説明・レビューできるか

これは実際の研究者が必ず状態ベクトルを順に調べると断定するものではありません。実際の小規模オラクルや状態準備サブルーチンを持ち込んでもらい、この表示が確認時間や説明のしやすさを改善するかを評価します。

`show_circuit()` は回路構造の確認、`show_circuit_and_state()` は小規模回路の状態変化の確認に使い分けます。どちらの用途が実際のワークフローに適合するかは、利用者との検証で確認します。

## 初期デモの範囲

- 対応範囲は1〜8量子ビットの小規模回路です。
- RX、RY、RZ、U1、未対応ゲート、アンチコントロール、保持できないclassical bit mappingは、別の意味で表示せず例外で停止します。
- GUI編集、`commit()`、VQE/QAOAのパラメータ式、20〜32量子ビットの性能保証は初期デモに含みません。

これで、**コード、回路構造、各ステップまでの状態をNotebook上で対応付けて確認する**、という最小デモは完了です。